# gpt architecture

this notebook implements the architecture behind a gpt-style model

In [1]:
import torch
import torch.nn as nn
import tiktoken

In [2]:
gpt_config_124m = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

In [6]:
class multi_head_attention(nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False,
    ):
        super().__init__()

        # ensures that the output dimension is divisible by the number of heads
        assert d_out % num_heads == 0.

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        # creates the trainable linear layers for the query, key, and value projections
        self.w_query = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )

        self.w_key = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )

        self.w_value = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )

        # creates the trainable linear layer for the output projection
        # mixes information from different heads
        self.out_proj = nn.Linear(
            d_out,
            d_out,
        )

        # creates a dropout layer to prevent overfitting
        self.dropout = nn.Dropout(
            dropout
        )

        # creates a causal mask to prevent attention to future tokens
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(
                    context_length,
                    context_length,
                ),
                diagonal=1,
            ),
        )

    def forward(self, x):

        # read the input shape
        batch_size, num_tokens, d_in = x.shape

        # compute the query, key, and value projections
        queries = self.w_query(x)
        keys = self.w_key(x)
        values = self.w_value(x)

        # split the query, key, and value projections into multiple heads
        queries = queries.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )

        keys = keys.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )

        values = values.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )

        # move the head dimension to the second position for matrix multiplication
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # calculate query-key dot products to get attention scores
        attention_scores = (
            queries
            @ keys.transpose(2, 3)
        )

        # apply the causal mask to prevent attention to future tokens
        causal_mask = self.mask.bool()[
            :num_tokens,
            :num_tokens
        ]

        # block the attention scores for future tokens by setting them to negative infinity
        attention_scores.masked_fill_(
            causal_mask,
            -torch.inf,
        )

        # scale and normalize the attention scores to get attention weights
        attention_weights = torch.softmax(
            attention_scores
            / keys.shape[-1] ** 0.5,
            dim=-1,
        )

        # apply dropout to the attention weights to prevent overfitting
        attention_weights = self.dropout(
            attention_weights
        )

        # combine the attention weights with the value projections to get the context vectors
        context_vectors = (
            attention_weights
            @ values
        )

        # move the head dimension back to the last position for the output projection
        context_vectors = (
            context_vectors.transpose(1, 2)
        )

        # recombine the multiple heads into a single vector for each token
        context_vectors = (
            context_vectors
            .contiguous()
            .view(
                batch_size,
                num_tokens,
                self.d_out,
            )
        )

        # mix information from different heads and project the context vectors back to the original input dimension
        context_vectors = self.out_proj(
            context_vectors
        )

        return context_vectors

In [7]:
class layer_norm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        # stores a small constant to prevent division by zero during normalization
        self.eps = 1e-5

        # creates trainable parameters for scaling and shifting the normalized output
        self.scale = nn.Parameter(
            torch.ones(emb_dim)
        )

        self.shift = nn.Parameter(
            torch.zeros(emb_dim)
        )

    def forward(self, x):

        # calculates the mean and variaance across each token's features
        mean = x.mean(
            dim=-1,
            keepdim=True,
        )

        variance = x.var(
            dim=-1,
            keepdim=True,
            unbiased=False,
        )

        # normalizes the input by subtracting the mean and dividing by the standard deviation
        normalized_x = (
            (x - mean)
            / torch.sqrt(
                variance + self.eps
            )
        )

        return (
            self.scale * normalized_x
            + self.shift
        )

In [8]:
class gelu(nn.Module):
    def __init__(self):
        super().__init__()

    # defines the activation function used in the feedforward network
    # it is a smooth approximation of the ReLU function
    def forward(self, x):
        coefficient = torch.sqrt(
            torch.tensor(
                2.0 / torch.pi,
                device=x.device,
            )
        )

        return (
            0.5
            * x
            * (
                1
                + torch.tanh(
                    coefficient
                    * (
                        x
                        + 0.044715
                        * torch.pow(x, 3)
                    )
                )
            )
        )

In [ ]:
class feed_forward(nn.Module):
    def __init__(self, config):
        super().__init__()

        # creates a feedforward network with two linear layers 
        # and a GELU activation function
        self.layers = nn.Sequential(
            nn.Linear(
                config["emb_dim"],
                4 * config["emb_dim"],
            ),
            gelu(),
            nn.Linear(
                4 * config["emb_dim"],
                config["emb_dim"],
            ),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
class transformer_block(nn.Module):
    def __init__(self, config):
        super().__init__()

        # creates the attention module
        self.attention = multi_head_attention(
            d_in=config["emb_dim"],
            d_out=config["emb_dim"],
            context_length=config[
                "context_length"
            ],
            dropout=config["drop_rate"],
            num_heads=config["n_heads"],
            qkv_bias=config["qkv_bias"],
        )

        # creates the feedforward module
        self.feed_forward = feed_forward(
            config
        )

        # creates two layer normalization modules 
        # onefor the attention and another for feedforward sublayers
        self.norm_1 = layer_norm(
            config["emb_dim"]
        )

        self.norm_2 = layer_norm(
            config["emb_dim"]
        )

        #creates residual connections with dropout to prevent overfitting
        self.shortcut_dropout = nn.Dropout(
            config["drop_rate"]
        )

    def forward(self, x):

        # save attention shortcut for residual connection
        shortcut = x

        # normalize before attention, apply attention, and apply dropout
        x = self.norm_1(x)
        x = self.attention(x)
        x = self.shortcut_dropout(x)

        # add the original input to the output of the attention sublayer
        x = x + shortcut

        # save feedforward shortcut for residual connection
        shortcut = x

        # normalize before feedforward, apply feedforward, and apply dropout
        x = self.norm_2(x)
        x = self.feed_forward(x)
        x = self.shortcut_dropout(x)

        # add a second residual connection 
        # from the input of the feedforward sublayer to its output
        x = x + shortcut

        return x

In [12]:
class gpt_model(nn.Module):
    def __init__(self, config):
        super().__init__()

        # creates a trainable embedding layer for the input tokens
        self.token_embedding = nn.Embedding(
            config["vocab_size"],
            config["emb_dim"],
        )

        # creates a trainable embedding layer for the positional encodings
        self.position_embedding = nn.Embedding(
            config["context_length"],
            config["emb_dim"],
        )

        # applies dropout after token and positional embeddings are combined
        self.embedding_dropout = nn.Dropout(
            config["drop_rate"]
        )

        # creates sequential container for the transformer blocks
        self.transformer_blocks = nn.Sequential(
            *[
                transformer_block(config)
                for _ in range(
                    config["n_layers"]
                )
            ]
        )

        # final normalization layer before the output head
        self.final_norm = layer_norm(
            config["emb_dim"]
        )

        # creates a linear layer 
        # to project the final hidden states to the vocabulary size
        self.output_head = nn.Linear(
            config["emb_dim"],
            config["vocab_size"],
            bias=False,
        )

    def forward(self, input_ids):

        # read the input dimensions
        batch_size, sequence_length = (
            input_ids.shape
        )

        # compute the token embeddings for the input token IDs
        token_embeddings = (
            self.token_embedding(
                input_ids
            )
        )

        # create a tensor of position indices for the input sequence
        positions = torch.arange(
            sequence_length,
            device=input_ids.device,
        )

        # compute the position embeddings for the input positions
        position_embeddings = (
            self.position_embedding(
                positions
            )
        )

        # look up the token and position embeddings, and add them together
        x = (
            token_embeddings
            + position_embeddings
        )

        # apply dropout to the combined embeddings to prevent overfitting
        x = self.embedding_dropout(x)

        # apply the transformer blocks to the input embeddings
        x = self.transformer_blocks(x)

        # normalize the final output of the transformer blocks 
        # before passing it to the output head
        x = self.final_norm(x)

        # project the final hidden states to the vocabulary size 
        # to get the logits for each token
        logits = self.output_head(x)

        return logits

In [25]:
# create the model and move it to the appropriate device (GPU or CPU)

torch.manual_seed(123)

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = gpt_model(gpt_config_124m)
model = model.to(device)

print("selected device:", device)

selected device: mps


In [26]:
# place model in evaluation mode

model.eval()

gpt_model(
  (token_embedding): Embedding(50257, 768)
  (position_embedding): Embedding(1024, 768)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): transformer_block(
      (attention): multi_head_attention(
        (w_query): Linear(in_features=768, out_features=768, bias=False)
        (w_key): Linear(in_features=768, out_features=768, bias=False)
        (w_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (feed_forward): feed_forward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): gelu()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm_1): layer_norm()
      (norm_2): layer_norm()
      (shortcut_dropout): Dropout(p=0.1, inplace=False)
    )
    (1): transformer_block(
    

In [27]:
# inspect number of parameters in the model

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("total parameters:", f"{total_parameters:,}")
print("trainable parameters:", f"{trainable_parameters:,}")

total parameters: 163,009,536
trainable parameters: 163,009,536


In [28]:
# create the tokenizer for the GPT-2 model using the tiktoken library

import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [29]:
# tokenize some input text

input_text = "Every effort moves you"

input_ids = tokenizer.encode(input_text)

input_tensor = torch.tensor(
    input_ids,
    dtype=torch.long,
)

input_tensor = input_tensor.unsqueeze(0)
input_tensor = input_tensor.to(device)

print("input text:", input_text)
print("token IDs:", input_ids)
print("input shape:", input_tensor.shape)
print("input device:", input_tensor.device)

input text: Every effort moves you
token IDs: [6109, 3626, 6100, 345]
input shape: torch.Size([1, 4])
input device: mps:0


In [30]:
# run the forward pass of the model with the tokenized input

with torch.no_grad():
    logits = model(input_tensor)

print("input shape:", input_tensor.shape)
print("logits shape:", logits.shape)

input shape: torch.Size([1, 4])
logits shape: torch.Size([1, 4, 50257])


In [32]:
# inspect the last-position logists for the next token prediction

last_position_logits = logits[:, -1, :]

print(
    "last-position logits shape:",
    last_position_logits.shape,
)


last-position logits shape: torch.Size([1, 50257])


In [33]:
# select the next token

next_token_id = torch.argmax(
    last_position_logits,
    dim=-1,
    keepdim=True,
)

print("next token ID:", next_token_id)

next token ID: tensor([[37532]], device='mps:0')


In [34]:
# decode the selected token

next_token_text = tokenizer.decode(
    next_token_id.squeeze(0).tolist()
)

print("predicted token:", repr(next_token_text))

predicted token: ' Ae'


In [35]:
# define the generation function to generate text using the trained model

def generate_text_simple(
    model,
    token_ids,
    max_new_tokens,
    context_size,
):
    for _ in range(max_new_tokens):
        conditioned_token_ids = token_ids[
            :, -context_size:
        ]

        with torch.no_grad():
            logits = model(
                conditioned_token_ids
            )

        last_position_logits = logits[
            :, -1, :
        ]

        next_token_id = torch.argmax(
            last_position_logits,
            dim=-1,
            keepdim=True,
        )

        token_ids = torch.cat(
            (token_ids, next_token_id),
            dim=1,
        )

    return token_ids

In [37]:
model.eval()

generated_token_ids = generate_text_simple(
    model=model,
    token_ids=input_tensor,
    max_new_tokens=10,
    context_size=gpt_config_124m["context_length"],
)

generated_text = tokenizer.decode(
    generated_token_ids.squeeze(0).tolist()
)

print("generated text:")
print(generated_text)

generated text:
Every effort moves you Aeiman Byeswickattributeometer inspector Normandy freezerigrate
